In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from configs.config_loader import Config
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)
notebook_config = Config("configs/app_notebook.yaml")


DOMAIN = "finance"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name, new_config=notebook_config)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("bench") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("bench") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("bench") / "data_dir/domains" / "base"
BASE_CODE_DIR      = Path("bench") / "domains" / "base"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/finance
生成数据目录：bench/data_dir/domains/finance
生成代码目录：bench/domains/finance
BASE 数据目录：bench/data_dir/domains/base


In [2]:
# 加载 BASE 域生成结果（作为 WORK 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()
with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")
print(f"BASE 表数量：{len(base_data)}")

加载完成
BASE 表数量：16


### 1. 生成 WORK Schema

In [4]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", base_schema)

print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

正在生成 finance Schema...
------------------------------------------------------------


我来一步步分析财务域的核心对象：

**核心对象抽象：**

1. **财务账户** - 企业银行账户、收款账户管理
2. **预算** - 部门/项目预算编制与执行
3. **费用报销单** - 各类报销（差旅、餐饮、交通、住宿、团建、招待等）
4. **报销明细行** - 报销单下的每笔费用
5. **发票** - 发票信息管理（采购发票、报销发票）
6. **付款申请** - 企业付款、采购付款、特殊付款
7. **付款执行** - 实际付款记录
8. **收款流水** - 收款记录
9. **退款单** - 退货退款处理
10. **员工借款** - 借款与还款
11. **结算单** - 供应商/客户结算
12. **资金调拨** - 内部资金转移
13. **个税记录** - 个税申报与测算
14. **税务申报** - 企业税务申报
15. **发票抬头** - 开票信息维护
16. **汇率** - 汇率记录
17. **预提/计提** - 费用预提计提
18. **报销政策** - 报销标准与制度
19. **财务咨询工单** - 财务服务台/热线

```json
{
    "finance_account": {
        "FACC001": {
            "finance_account_ref": "FACC001",
            "account_name": "...",
            "account_number": "...",
            "bank_name": "...",
            "bank_branch": "...",
            "swift_code": null,
            "currency": "CNY",
            "account_type": "settlement",
            "account_purpose": "general",
            "account_status": "acti

### 2. 生成 WORK Data Models

In [5]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt = f.read()

models_prompt = models_prompt.replace("{base_models}", base_models)
models_prompt = models_prompt.replace("{domain_schema}", domain_schema)

print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
models_response = ""
for chunk in llm.llm.stream(models_prompt):
    content = chunk.content
    models_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

models_code = extract_code_block(models_response)
is_valid, err = verify_python_syntax(models_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {err}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), models_code)
print(f"✓ Models 已保存到: {models_path}")

正在生成 finance Data Models...
------------------------------------------------------------


```python
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

from bench.domains.base.models import OfficeDB


# --- Enums as Literal types ---

FinanceAccountType = Literal["settlement", "payroll", "petty_cash", "escrow", "investment", "tax"]
FinanceAccountPurpose = Literal["general", "operating", "capital", "reserve"]
FinanceAccountStatus = Literal["active", "frozen", "closed"]
Currency = Literal["CNY", "USD", "EUR", "GBP", "JPY", "HKD", "SGD", "KRW", "TWD"]

BudgetType = Literal["department", "project", "company", "special"]
BudgetScopeType = Literal["department", "team", "company", "project"]
BudgetStatus = Literal["draft", "submitted", "approved", "rejected", "closed", "frozen"]
FiscalPeriod = Literal["Q1", "Q2", "Q3", "Q4", "H1", "H2", "annual"]

ExpenseCategory = Literal[
    "travel", "accommodation", "meals", "transportation", "office_supplies",


### 3. Schema/Model一致性检查

In [6]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

正在验证 Schema 和 Model 的一致性...
我来逐一检查这三点。

## 检查过程

### 1. Data model 中的字段和类型与 Schema 定义一致

逐个对比 Schema 和 Data model：

**FinanceAccount** ✅ 一致

**Budget** ✅ 一致

**BudgetLine** ✅ 一致

**ExpenseReport** ✅ 一致

**ExpenseLine** ✅ 一致

**Invoice** ✅ 一致

**InvoiceHeader** ✅ 一致

**PaymentRequest** ✅ 一致

**PaymentExecution** ✅ 一致

**ReceiptRecord** ✅ 一致

**Refund** ✅ 一致

**EmployeeLoan** ✅ 一致

**LoanRepayment** ✅ 一致

**Settlement** ✅ 一致

**SettlementLine** ✅ 一致

**FundTransfer** ✅ 一致

**ExchangeRate** ✅ 一致

**TaxFiling** ✅ 一致

**EmployeeTaxRecord** ✅ 一致 - `special_additional_deductions` 在 Schema 中是嵌套对象，Data model 中是 `Dict[str, float]`，合理。

**AccrualEntry** ✅ 一致

**ExpensePolicy** ✅ 一致

**FinanceTicket** ✅ 一致

**CreditApplication** ✅ 一致

**CreditRepaymentSchedule** ✅ 一致

**StockTradeOrder** ✅ 一致

**FinanceApprovalFlow** ✅ 一致

### 2. Data model 中的字段和类型在不同位置保持一致

- `currency` 字段：在所有模型中都使用 `Currency` 类型 ✅
- `settlement_currency` 在 `PaymentExecution` 中是 `Optional[str]` 而非 `Optional[Currency]`，`target_cur

### 4. 生成数据脚本（Faker，主流程）

In [7]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 IT 域的外键依赖（不含 BASE 表），推导生成顺序
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"IT 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

local_llm = LLMClient("gpt-5.4-0305-global", 128000, new_config=notebook_config)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

IT 域表生成顺序（拓扑排序，共 26 张表）：
   1. financeaccounts
   2. budgets
   3. invoices
   4. invoiceheaders
   5. paymentexecutions
   6. receiptrecords
   7. refunds
   8. employeeloans
   9. loanrepayments
  10. settlements
  11. fundtransfers
  12. exchangerates
  13. taxfilings
  14. employeetaxrecords
  15. accrualentries
  16. expensepolicies
  17. financetickets
  18. creditapplications
  19. creditrepaymentschedules
  20. stocktradeorders
  21. financeapprovalflows
  22. budgetlines
  23. expensereports
  24. paymentrequests
  25. expenselines
  26. settlementlines

FK 依赖映射：
  financeaccounts: [('finance_account_ref', 'finance_accounts'), ('department_ref', 'departments'), ('managed_by_ref', 'managed_bies')]
  budgets: [('budget_ref', 'budgets'), ('scope_ref', 'scopes'), ('parent_budget_ref', 'parent_budgets'), ('approved_by_ref', 'approved_bies')]
  budgetlines: [('budget_line_ref', 'budget_lines'), ('budget_ref', 'budgets')]
  expensereports: [('expense_report_ref', 'expense_reports'), 

### 3.5 执行数据脚本，生成 db.json

In [11]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)

print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（WORK 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ WORK Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 26 张表
  - finance_account: 20 条
  - budget: 24 条
  - invoice: 200 条
  - invoice_header: 20 条
  - payment_execution: 200 条
  - receipt_record: 200 条
  - refund: 200 条
  - employee_loan: 200 条
  - loan_repayment: 200 条
  - settlement: 200 条
  - fund_transfer: 200 条
  - exchange_rate: 40 条
  - tax_filing: 200 条
  - employee_tax_record: 200 条
  - accrual_entry: 200 条
  - expense_policy: 30 条
  - finance_ticket: 200 条
  - credit_application: 200 条
  - credit_repayment_schedule: 200 条
  - stock_trade_order: 200 条
  - finance_approval_flow: 200 条
  - budget_line: 200 条
  - expense_report: 200 条
  - payment_request: 200 条
  - expense_line: 200 条
  - settlement_line: 200 条
✓ 外键完整性验证通过

✓ WORK Database 已保存到: bench/data_dir/domains/finance/db.json


In [12]:
# Pydantic 模型验证：合并 BASE + WORK 数据后做完整校验
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from bench.domains.finance.models import FinanceDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = json.load(f)

merged_raw = {**base_data, **db_raw}

try:
    work_db = FinanceDB.model_validate(merged_raw)
    print("✓ WorkDB 模型验证通过")
    print(work_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ WorkDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_finance_accounts': 20, 'num_budgets': 24, 'num_budget_lines': 200, 'num_expense_reports': 200, 'num_expense_lines': 200, 'num_invoices': 200, 'num_invoice_headers': 20, 'num_payment_requests': 200, 'num_payment_executions': 200, 'num_receipt_records': 200, 'num_refunds': 200, 'num_employee_loans': 200, 'num_loan_repayments': 200, 'num_settlements': 200, 'num_settlement_lines': 200, 'num_fund_transfers': 200, 'num_exchange_rates': 40, 'num_tax_filings': 200, 'num_employee_tax_records': 200, 'num_accrual_entries': 200, 'num_expense_policies': 30, 'num_finance_tickets': 200, 'num_credit_applications': 200, 'num_credit_repayment_schedules': 200, 'num_stock_trade_orders': 200, 'num_finance_approval_flows': 200}


### 4. 生成 WORK Tools

In [13]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# WORK 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
work_function_description = """""财务税务": [
    {
      "title": "个税申报",
      "definition": "用户希望获得个人所得税申报相关的指导和帮助，包括税务政策解读、申报流程指引、税款计算、专项附加扣除填报、申报表格填写以及相关税务问题咨询等。"
    },
    {
      "title": "个税测算",
      "definition": "用户希望人工智能协助计算个人所得税相关费用，包括工资薪金、年终奖、劳务报酬等不同收入类型的税额计算，以及专项附加扣除、税率适用、预扣预缴等税务筹划和合规计算。"
    },
    {
      "title": "中银国际股票交易",
      "definition": "用户希望通过人工智能助手获取股票市场信息、执行交易指令、监控投资组合表现，包括实时股价查询、交易执行、风险分析、投资建议生成等专业金融服务支持。"
    },
    {
      "title": "支付管理",
      "definition": "用户希望通过人工智能助手处理与支付相关的各类办公事务，包括支付流程审批、费用报销管理、供应商付款跟踪、支付数据统计分析、发票处理以及支付合规性检查等财务支付环节的自动化处理和智能化管理。"
    },
    {
      "title": "金融信贷服务",
      "definition": "用户希望获得金融信贷相关的专业支持和服务，包括贷款产品咨询、信贷政策解读、风险评估分析、贷款申请流程指导、还款计划制定、信用报告分析等，以及生成相关的金融文档和报表。"
    },
    {
      "title": "采购付款管理",
      "definition": "用户希望通过人工智能助手高效管理采购流程中的付款环节，包括供应商付款申请审批、付款计划制定与跟踪、发票核对与处理、付款状态查询、逾期付款提醒、供应商账期管理等，以确保采购付款的及时性、准确性和合规性。"
    },
    {
      "title": "退货退款",
      "definition": "用户需要处理客户的退货退款申请，包括审核退货原因、计算退款金额、生成退款单据、跟踪退款流程状态，以及制作相关的退货退款报表和通知文档。"
    },
    {
      "title": "采购发票管理",
      "definition": "用户希望通过人工智能助手高效管理采购发票的全生命周期，包括发票信息录入、自动识别发票内容、核对采购订单与发票的匹配性、跟踪发票审批流程、管理付款状态，以及生成采购发票统计报表和分析。"
    },
    {
      "title": "报销流程指引",
      "definition": "用户希望了解和掌握公司报销相关的操作流程、政策规定和注意事项，包括报销申请的提交步骤、所需材料清单、审批流程、时间节点、报销标准以及常见问题的解决方案。"
    },
    {
      "title": "报销政策咨询",
      "definition": "用户希望了解公司报销相关的政策规定、流程要求和标准限额，包括差旅费、餐费、交通费等各类费用的报销条件、所需材料、审批流程以及相关注意事项。"
    },
    {
      "title": "报销单据管理",
      "definition": "用户希望通过人工智能助手高效处理报销相关的各类单据，包括费用录入、单据分类整理、审批流程跟踪、报销标准核验、发票识别录入、以及报销进度查询等，以简化报销流程并提高财务管理效率。"
    },
    {
      "title": "交通费报销",
      "definition": "用户希望人工智能协助处理交通费用的报销流程，包括费用记录整理、报销单据填写、费用分类统计、报销流程指导、单据格式检查以及相关财务制度咨询等报销相关事务。"
    },
    {
      "title": "住宿费报销",
      "definition": "用户希望人工智能协助处理住宿费用的报销流程，包括住宿费报销单据的填写、费用标准查询、报销政策解读、单据审核要点提醒、报销流程指导等，以提高报销效率和准确性。"
    },
    {
      "title": "餐饮费报销",
      "definition": "用户希望人工智能协助处理餐饮费用的报销流程，包括费用分类、发票信息提取、报销单据填写、合规性检查、审批流程跟踪等，以提高报销效率和准确性。"
    },
    {
      "title": "其他差旅费用",
      "definition": "用户需要处理和管理除住宿、交通外的各类差旅相关费用，包括餐费、通讯费、签证费、保险费等杂项支出的录入、分类、审批和报销流程管理。"
    },
    {
      "title": "入职/离职报销",
      "definition": "用户希望人工智能协助处理员工入职或离职过程中涉及的各类费用报销事务，包括报销流程指导、报销单据填写、费用标准查询、审批流程跟踪以及相关财务政策解答等。"
    },
    {
      "title": "团建/培训报销",
      "definition": "用户希望人工智能协助处理团建活动或培训项目的费用报销流程，包括费用统计、报销单据整理、审批流程跟进、预算核对以及相关财务文档的生成和管理。"
    },
    {
      "title": "招待费报销",
      "definition": "用户希望人工智能协助处理招待费用的报销流程，包括费用分类、报销单据填写、合规性检查、审批流程指导以及相关财务制度咨询等报销相关事务。"
    },
    {
      "title": "用车报销",
      "definition": "用户希望通过人工智能助手处理与公务用车相关的费用报销流程，包括录入用车信息、计算报销金额、生成报销单据、查询报销标准和政策、跟踪审批状态等，以提高报销效率和准确性。"
    },
    {
      "title": "发票类型选择",
      "definition": "用户需要人工智能协助识别、分类或选择合适的发票类型，包括增值税专用发票、普通发票、电子发票等不同类型的判断和选择，以确保财务处理的准确性和合规性。"
    },
    {
      "title": "发票抬头信息",
      "definition": "用户需要查询、管理或设置开具发票时的抬头信息，包括公司名称、纳税人识别号、地址电话、开户行及账号等基本信息，以确保发票开具的准确性和合规性。"
    },
    {
      "title": "发票处理问题",
      "definition": "用户希望人工智能协助处理发票相关的各种业务场景，包括发票信息的识别提取、发票真伪验证、发票分类整理、报销流程咨询、税务合规检查以及发票数据的统计分析等财务管理工作。"
    },
    {
      "title": "企业付款流程管理",
      "definition": "用户希望通过人工智能助手优化和管理企业的付款审批流程，包括付款申请的创建、审批流程的跟踪、付款状态的查询、审批节点的提醒，以及付款相关文档的生成和归档管理。"
    },
    {
      "title": "具体付款问题咨询",
      "definition": "用户希望获得关于付费服务、账单、支付方式、退款政策等财务相关问题的解答和指导，包括查询付款状态、解决支付异常、了解收费标准或处理发票报销等办公场景中的财务咨询需求。"
    },
    {
      "title": "特付管理",
      "definition": "用户的核心目的是对企业内部的特殊付款、费用报销、预算审批等财务流程进行管理和监控，包括查询特付申请状态、审批流程跟踪、费用分类统计、预算执行情况分析以及相关财务单据的处理和归档。"
    },
    {
      "title": "结算单管理",
      "definition": "用户希望对各类结算单据进行系统化管理，包括创建、录入、审核、查询、统计和归档结算单，以及跟踪结算状态、生成结算报表、处理结算异常和维护供应商/客户结算信息等操作。"
    },
    {
      "title": "预提与计提",
      "definition": "用户希望人工智能协助进行财务预算编制、成本预估或收入预测等前瞻性财务规划工作，包括项目预算制定、部门费用预提、季度/年度财务预测等。\n\n计提：用户需要人工智能协助进行财务费用的确认和分摊处理，包括折旧计提、税费计提、准备金计提、成本分摊等会计核算工作。"
    },
    {
      "title": "收款账户管理",
      "definition": "用户希望对企业的收款账户信息进行统一管理和维护，包括添加、修改、删除收款账户，查询账户状态和余额，设置账户权限，以及生成收款账户相关的报表和统计分析。"
    },
    {
      "title": "收款流水处理",
      "definition": "用户希望人工智能协助处理和分析企业的收款流水数据，包括对收款记录进行分类整理、异常检测、对账核验、生成收款统计报表，以及识别逾期款项等财务管理任务。"
    },
    {
      "title": "退款处理",
      "definition": "用户希望人工智能协助处理客户退款相关的业务流程，包括退款申请的审核、退款金额计算、退款流程跟踪、退款文档生成、以及与财务系统的数据对接等工作。"
    },
    {
      "title": "资金查询",
      "definition": "用户希望快速获取企业或项目的资金相关信息，包括查询账户余额、资金流水、预算执行情况、费用明细、投资回报数据等财务资金状况，以支持日常财务决策和资金管理工作。"
    },
    {
      "title": "资金调拨",
      "definition": "用户希望通过人工智能助手处理企业内部资金在不同部门、项目或账户间的转移分配工作，包括制定调拨方案、生成调拨申请文档、跟踪调拨进度、核算资金流向以及生成相关的财务报表和审批材料。"
    },
    {
      "title": "汇率与外汇",
      "definition": "用户需要获取实时或历史汇率数据、外汇市场信息，进行货币换算计算，或基于汇率变动进行财务分析和风险评估，以支持国际业务决策、财务报表编制或投资规划。"
    },
    {
      "title": "财务权限申请",
      "definition": "用户需要申请或管理财务系统的访问权限，包括申请查看财务报表、预算数据、成本中心信息等权限，或为团队成员申请相应的财务模块操作权限，以及跟踪权限申请的审批状态。"
    },
    {
      "title": "财务角色管理",
      "definition": "用户的核心目的是对财务部门的人员权限、职责分工和角色配置进行系统化管理，包括设置不同财务岗位的系统访问权限、数据查看范围、审批流程权限，以及财务团队成员的角色分配、职责界定和权限变更等操作。"
    },
    {
      "title": "财务预算管理",
      "definition": "用户希望通过人工智能助手进行预算制定、执行监控和分析优化，包括预算编制模板生成、费用分类统计、预算执行进度跟踪、差异分析报告、成本控制建议以及财务数据可视化展示等功能。"
    },
    {
      "title": "税务信息查询",
      "definition": "用户希望快速获取与税务相关的各类信息，包括税率查询、税收政策解读、纳税申报要求、税务优惠政策、发票管理规定、企业所得税计算方法等，以支持日常财务工作和税务合规管理。"
    },
    {
      "title": "发票税务处理",
      "definition": "用户希望人工智能协助处理与发票和税务相关的各类工作，包括发票信息识别与录入、税率计算、税务申报表填写、发票真伪验证、进销项发票管理、税务合规性检查等财税业务流程的自动化处理。"
    },
    {
      "title": "税务申报",
      "definition": "用户希望人工智能协助处理企业或个人的税务申报相关工作，包括税务表格填写、税收计算、申报材料准备、税务政策咨询、申报流程指导以及相关财务数据整理等税务合规事务。"
    },
    {
      "title": "财务制度咨询",
      "definition": "用户希望了解和查询公司的财务管理制度、流程规范和相关政策，包括报销制度、预算管理、费用审批流程、财务报告要求、税务政策等具体条款和执行标准。"
    },
    {
      "title": "财务服务台查询",
      "definition": "用户希望快速获取财务相关信息和服务支持，包括查询财务报表数据、预算执行情况、费用报销状态、发票开具进度、账务处理结果、财务政策解释以及获得财务业务流程指导等服务。"
    },
    {
      "title": "财务热线查询",
      "definition": "用户希望快速获取财务相关信息和解答财务问题，包括查询财务政策、报销流程、预算状态、费用标准、财务审批进度、发票处理状态等财务事务的咨询和处理。"
    },
    {
      "title": "员工借款",
      "definition": "用户希望处理员工借款相关的业务流程，包括借款申请表单的生成、借款审批流程的跟踪、借款记录的查询统计、还款提醒通知以及借款相关文档的自动生成等财务管理事务。"
    },
    {
      "title": "财务报账管理",
      "definition": "用户希望通过人工智能助手简化和优化财务报销流程，包括费用单据的识别录入、报销规则校验、审批流程跟踪、财务数据统计分析，以及生成各类财务报表和凭证，提高报账效率和准确性。"
    }
  ]
"""

with open(BASE_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", work_function_description)

print(f"正在生成 {DOMAIN} Tools...")
print("-" * 60)
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 finance Tools...
------------------------------------------------------------


```python
"""Toolkit for the finance/work domain, extending OfficeTools."""

from typing import Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.domains.base.tools import OfficeTools
from bench.environment.toolkit import ToolType, is_tool
from bench.domains.work.models import (
    FinanceDB as WorkDB,
    FinanceAccount,
    Budget,
    BudgetLine,
    ExpenseReport,
    ExpenseLine,
    Invoice,
    InvoiceHeader,
    PaymentRequest,
    PaymentExecution,
    ReceiptRecord,
    Refund,
    EmployeeLoan,
    LoanRepayment,
    Settlement,
    SettlementLine,
    FundTransfer,
    ExchangeRate,
    TaxFiling,
    EmployeeTaxRecord,
    AccrualEntry,
    ExpensePolicy,
    FinanceTicket,
    CreditApplication,
    CreditRepaymentSchedule,
    StockTradeOrder,
    FinanceApprovalFlow,
)


class WorkTools(OfficeTools):
    """Toolkit for the WORK domain, extending Offic